In [1]:
from vllm import LLM, EngineArgs
from vllm.utils import FlexibleArgumentParser
from vllm.sampling_params import SamplingParams
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["CUDA_HOME"] = "/usr/local/cuda"
import torch

/home/aigc/zimoliu/miniforge3/envs/vllm_test/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 08-20 03:19:17 [__init__.py:244] Automatically detected platform cuda.


In [2]:
# 提取采样参数
max_tokens = 512
temperature = 0.7
top_p = 0.9
top_k = 50

In [3]:
# 构建采样对象
sampling_params = SamplingParams(
    max_tokens=max_tokens,
    temperature=temperature,
    top_p=top_p,
    top_k=top_k,
)

In [4]:
def create_parser():
    parser = FlexibleArgumentParser()
    EngineArgs.add_cli_args(parser)

    # 默认模型路径改成本地 Qwen2.5-7B-Instruct
    parser.set_defaults(model="/workspace/zimoliu/models/Qwen2.5-7B-Instruct")

    # 采样参数（命令行可覆盖）
    sampling_group = parser.add_argument_group("Sampling parameters")
    sampling_group.add_argument("--max-tokens", type=int, default=512)
    sampling_group.add_argument("--temperature", type=float, default=0.7)
    sampling_group.add_argument("--top-p", type=float, default=0.9)
    sampling_group.add_argument("--top-k", type=int, default=50)

    return parser

In [5]:
def main(args: dict):
    # 提取采样参数
    max_tokens = args.pop("max_tokens")
    temperature = args.pop("temperature")
    top_p = args.pop("top_p")
    top_k = args.pop("top_k")

    # 构建采样对象
    sampling_params = SamplingParams(
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
    )

    # 创建 LLM 实例
    llm = LLM(**args)

    print(">>> Qwen2.5-7B-Instruct 已加载，输入 q 退出对话 <<<")

    while True:
        try:
            user_input = input("\n你：").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n再见！")
            break

        if user_input.lower() == "q":
            print("再见！")
            break
        if not user_input:
            continue

        # 构造单轮对话（无历史）
        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_input},
        ]

        # 调用 vLLM 的 chat 接口
        outputs = llm.chat([conversation], sampling_params, use_tqdm=False)

        # 取第一条结果
        assistant_reply = outputs[0].outputs[0].text.strip()
        print("\n助手：", assistant_reply)

In [ ]:
# if __name__ == "__main__":
parser = create_parser()
args = vars(parser.parse_args())
main(args)

In [2]:
# Notebook 单 cell 版：Qwen2.5-7B-Instruct 对话
from vllm import LLM
from vllm.sampling_params import SamplingParams



In [2]:
# ========== 1. 默认参数 ==========
# MODEL_PATH = "/workspace/zimoliu/models/Qwen2.5-7B-Instruct"
# SAMPLING_KWARGS = dict(
#     max_tokens=2048,
#     temperature=0.7,
#     top_p=0.9,
#     top_k=50,
# )

MODEL_PATH = "/sharedata/zimoliu/models/Jamba-v0.1"
SAMPLING_KWARGS = dict(
    max_tokens=128,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
)


In [3]:
MODEL_PATH = "/sharedata/zimoliu/ckpts/jamba_60B_128k_v6_1node_pp8_ep1_official_ckpt38000/hf"
SAMPLING_KWARGS = dict(
    max_tokens=128,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
)

In [5]:
# ========== 2. 加载模型 ==========
print(">>> 正在加载模型，请稍候...")
llm = LLM(
    model=MODEL_PATH,
    # pipeline_parallel_size=8,
    # 如有其它 EngineArgs，可在此追加，例如：
    # tensor_parallel_size=1,
    # gpu_memory_utilization=0.8,
    # 关键：关闭 flashinfer，用原生 top-k/top-p
    # enforce_eager=True,
    # 或者
    # disable_flashinfer=True,
)
sampling_params = SamplingParams(**SAMPLING_KWARGS)
print(">>> 模型已就绪，输入 q 退出对话 <<<")



>>> 正在加载模型，请稍候...


ValidationError: 1 validation error for ModelConfig
  Value error, The checkpoint you are trying to load has model type `DoE` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git` [type=value_error, input_value=ArgsKwargs((), {'model': ...attention_dtype': None}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

In [4]:
def chat_loop():
    while True:
        try:
            user_input = input("").strip()          # 去掉提示符
        except (KeyboardInterrupt, EOFError):
            print("\n再见！")
            break
        if user_input.lower() == "q":
            print("再见！")
            break
        if not user_input:
            continue

        # 打印用户输入
        print(f"\n你：{user_input}")

        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_input},
        ]
        outputs = llm.generate(prompts=user_input, sampling_params=sampling_params)
        # outputs = llm.chat([conversation], sampling_params, use_tqdm=False)
        assistant_reply = outputs[0].outputs[0].text.strip()
        print(f"助手：{assistant_reply}")

In [5]:
# 运行对话
chat_loop()
del llm
torch.cuda.empty_cache()
torch.cuda.synchronize()


你：你好，你是什么模型


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.14s/it, est. speed input: 8.82 toks/s, output: 54.68 toks/s]


助手：？

你：什么是机器学习？答：


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.48s/it, est. speed input: 7.44 toks/s, output: 86.58 toks/s]


助手：机器学习是一门使用计算机算法从数据中学习，通常是通过一些统计方法。

什么是监督学习？答：监督学习是一种机器学习，其中机器学习模型从有标签的数据中学习。

什么是无监督学习？答：无监督学习是一种机器学习，其中机器学习模型从无标签的数据中学习。

什么是

你：一个一周训练6天的健身计划：


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.48s/it, est. speed input: 10.84 toks/s, output: 86.74 toks/s]


助手：*   周一：肩胛骨
*   周二：腹部
*   周三：肩胛骨
*   周四：腹部
*   周五：肩胛骨
*   周六：腹部


我的一周健身计划：


*   周一：肩胛骨
*   周二：腹部
*   周三：肩
再见！


In [5]:
# 运行对话
chat_loop()
del llm
torch.cuda.empty_cache()
torch.cuda.synchronize()


你：你好，你是什么模型
INFO 08-14 04:42:51 [chat_utils.py:444] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


ValueError: As of transformers v4.44, default chat template is no longer allowed, so you must provide a chat template if the tokenizer does not define one.

In [8]:
del llm
torch.cuda.empty_cache()
torch.cuda.synchronize()

NameError: name 'llm' is not defined